# CRUD, formulaires, gestions d'utilisateurs


In [33]:
# on restaure notre base de données en remplaçant la BDD par son backup 
!cp ./richelieu.db.bak ./richelieu.db

On sait maintenant faire **toutes les opérations CRUD via une application Flask**. On peut donc utiliser ensemble Flask, SQLAlchemy, WTForms et les templates Jinja pour faire presque tout ce qui est attendu d'une appli Web.

Pour ce dernier cours, on va voir:
- comment **créer et gérer des comptes utilisateurs**
- comment **tester une appli Flask**
- comment **écrire une API**


---

# Les `users`

![db schema](./img/db_schema.png)

Votre regard aguisé aura remarqué que pour le moment, on a pas modélisé la table `user`. Elle sert à:
- ajouter de la gestion d'utilisateur.ice.s sur le site (créer un compte utilisateur et se connecter)
- limiter l'accès à certaines pages aux utilisateur.ice.s connecté.e.s. Par exemple, **on ne pourra modifier la base de données que si on est connecté.e**.

On remarque aussi que `user` n'est liée à aucune autre table: elle sert seulement à stocker les utilisateur.ice.s.

## Le modèle de `User`

Un compte utilisateur, c'est simplement une ligne de la table `user`.

**Voici notre table `User`** de base. 

```py
class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
```

Normalement, il existe déjà un compte utilisateur qui a pour mail `admin@mail.com` et pour mot de passe `admin`.


In [34]:
# on recrée notre appli, et notre modèle `User`
from typing import List, Optional
from pathlib import Path

from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from werkzeug.security import generate_password_hash
from sqlalchemy import ForeignKey
from sqlalchemy.orm import Mapped, mapped_column

path_to_db = Path("./richelieu.db").absolute()
print(path_to_db)

APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)

class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]

# on fait une requête pour voir les utilisateur.ice.s:
with app.app_context():
    users = db.session.execute(db.select(User)).scalars().all()

    print("Nombre d'utilisateur.ice.s:", len(users))

    user = users[0]
    print(f"nom: {user.user_name}, mail:{user.user_mail}, mdp: {user.user_password}")

/home/paul/Documents/cours/tnah_devapp/richelieu.db
Nombre d'utilisateur.ice.s: 1
nom: admin, mail:admin@mail.com, mdp: scrypt:32768:8:1$ttz8dh2PA9LayiiT$fee7c74bf665c4ba8fd2937297a0c968308d39db17394742ec52a7fd67ff05cd3cb62314a4c65fba0e74a6bd89d1cec9be248213f6fb06c71d1cf0dab7db2651


## Sécurité: stocker les mots de passe

Le mot de passe affiché au dessus (`scrypt:...`), c'est le mot de passe tel qu'il est enregistré dans la base de données. Cette énorme chaîne de caractères **n'est pas le mot de passe tel que définit pour l'utilisateur, mais un `hash` du mot de passe** (le "vrai" mot de passe utilisé pour se connecter, c'est `admin`). 

### Hashage ?

Pourquoi ce qui est stocké en base n'est pas le vrai mot de passe ? Parce que **⚠️⚠️⚠️ ON NE STOCKE JAMAIS UN MOT DE PASSE EN BRUT DANS UNE BASE DE DONNÉES ⚠️⚠️⚠️**. Quand on créée un compte en ligne, **c'est un hash du mot de passe qui est stocké**.

**Un hash, c'est une chaîne de caractères** générée par une "fonction de hashage" à partir d'une chaîne de caractères en entrée. Ce hash est:

- **irréversible**: si `tartempion` est hashé en `xff3eoa42`, il est impossible de retrouver `tartempion` à partir de `xff3eoa42`.
- **unique** à une valeur d'entrée: le hash `xff3eoa42` ne peut être obtenu qu'avec l'entrée `tartempion`.

L'intérêt, c'est d'éviter de stocker des données sensibles: si il y a une fuite de données, les mots de passe de vos utilisateurs ne seront pas compromis. Par exemple, votre ordinateur ne sait pas quel est votre mot de passe: quand vous vous connectez, il calcule le hash de votre mot de passe et vérifie si cela correspond au hash qu'il a enregistré. Pour les mots de passe d'une application, c'est pareil !

(Mais par contre, il arrive qu'un algorithme de hashage soit "cracké": on peut trouver une manière d'obtenir le mot de passe à partir de son hash. Dans ce cas, il s'agit d'une grosse faille de sécurité.)

### Hasher un mot de passe

Écrire un algo de hashage dépasse très largement mes compétences. Fort heureusement, **Werkzeug (librairie sous-couche de Flask) a deux fonctions qui gèrent le hashage** pour nous:

- `generate_password_hash()`: créer un hash à partir d'un mot de passe (utilisé pour définir un mot de passe)
- `check_password_hash()`: vérifier que le mot de passe fourni est correct (utilisé quand on se connecte)


In [35]:
from werkzeug.security import generate_password_hash, check_password_hash

mdp = "tartempion"

mdp_hash = generate_password_hash(mdp)
print("le hash est:", mdp_hash)

print("check_password_hash avec le bon mdp:", check_password_hash(mdp_hash, mdp))
print("check_password_hash avec le mauvais mdp:", check_password_hash(mdp_hash, "ceci est une vilaine tentative d'intrusion"))

le hash est: scrypt:32768:8:1$2wABDTdj5q1Vz6Cw$a40d98eb4c3d39c5a94d13569a20203517f09716f677e9eb88588e243857196cbcdda003a98a2d62d635e4f86436f0b21ad7f2081e7c823caf0b537635c6c54c
check_password_hash avec le bon mdp: True
check_password_hash avec le mauvais mdp: False



---

# Créer un compte utilisateur depuis l'application

On a vu au dernier cours comment faire des *create* depuis l'application:
- on crée un modèle de base de données `User`
- on lui ajoute une méthode `User.create()` qui permette de créer un nouveau compte utilisateur
- on crée un formulaire WTForms nommé `UserCreateForm` pour créer un nouveau `User` depuis l'application
- on crée une template Jinja `user_create.html`
- on crée une route `user_create` qui permette de créer un `User`.

Dans l'application [`create_user`](./apps/s6/create_user/), on va donc créer ou modifier les fichiers suivants:

```txt
create/
└── app
    ├── app.py        # il faudra importer les routes de routes/users.py
    ├── models
    │   ├── forms.py  # on ajoute `UserCreateForm`
    │   └── users.py  # nouveau fichier qui stocke le modèle SQLAlchemy pour les `Users`
    ├── routes
    │   └── users.py  # nouveau fichier stockant toutes les routes relatives à la gestion d'utilisateurices
    └── templates
        └── pages
            └── user_create.html  # template HTML avec un formulaire pour créer des utilisateurices
```

## Créer un utilisateur depuis `User`:  `create`

Pour créer un `User` et le sauvegarder en base dans notre application, on va:
- **créer un fichier [`app/models/users.py`](./apps/s6/create_user/app/models/users.py)** qui contient notre modèle `User`
- **créer une fonction `User.create`** qui gère la création d'utilisateurs. `create` prend en paramètres `user_name`, `user_mail` et `user_password` et créé l'`user` si il n'existe pas déjà.

Regardons bien le code ci-dessous:

```py
class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]

    @staticmethod
    def create(user_mail: str, user_name: str, user_password: str) -> Tuple[bool, Union["User", str]]:
        """
        créer un nouveau user.

        notre fonction retourne:
        - (True, User) en cas de succès 
        - (False, <message d'erreur>) en cas d'erreur
        donc, le 1er item permet de savoir si l'insertion a fonctionné 
        """

        # on vérifie si il existe un autre `user` avec le même mail
        existing_user = db.session.execute(
            db.select(User).filter(User.user_mail == user_mail)
        ).scalars().all()
        # l'user existe => on n'insère pas et on retourne un message d'erreur
        if len(existing_user):
            return False, f"Un utilisateur existe déjà pour le mail: {user_mail}"

        # si l'user n'existe pas, on le crée. remarquez l'utilisation de `generate_password_hash`
        new_user = User(
            user_name=user_name,
            user_mail=user_mail,
            user_password=generate_password_hash(user_password)
        ) 

        # pour finir, on insère l'user
        try:
            db.session.add(new_user)
            db.session.commit()
            return True, new_user
        except Exception as e:
            # en cas d'erreur au moment de l'insert, on retourne False et le message d'erreur de l'appli
            print(e)
            return False, "Erreur à la création du compte utilisateur"
```

Remarques:
- `User.create()` fonctionne **globalement globalement comme `Iconography.create`**: on reçoit les valeurs à sauvegarder en argument de la fonction, on crée l'objet (ici `User`), on l'insère dans un `try...except`
- **principale différence**: on ajoute une requête SQLAlchemy pour vérifier si un `user` n'existe pas avec le même mail

## Le formulaire `UserCreateForm`

On l'a dit, [`app/models/forms.py`](`./apps/s6/create_user/app/models/forms.py`) stockera le nouveau formulaire `UserCreateForm`.

Pour rappel, pour créer un nouveau `user` on a besoin d'un `user_mail`, d'un `user_name` et d'un `user_password`. Donc, **notre formulaire aura 3 champs obligatoires**.

**Voici `UserCreateForm`**, qui permet de créer un formulaire.

```py
from flask_wtf import FlaskForm
from wtforms import StringField, PasswordField
from wtforms.validators import DataRequired, Email, Length

class UserCreateForm(FlaskForm):
    user_name = StringField("Nom d'utilisateur.ice", validators=[DataRequired(), Length(max=50)])
    user_mail = StringField("Email", validators=[DataRequired(), Email()])
    user_password = PasswordField("Password", validators=[DataRequired(), Length(min=5, max=50)])
```

## La template `user_create.html`

La template HTML [`app/templates/pages/user_create.html`](./apps/s6/create_user/app/templates/pages/user_create.html) permet de **créer une interface utilisateur pour le formulaire**, via une template HTML. Voici son contenu:

```html
{% extends "base.html" %}

{% block title_extra %}| Créer un compte utilisateur{% endblock %}

{% block main_content %}
    <h1 class="title">Créer un compte utilisateur</h1>

    <form method="POST" action="{{ url_for('user_create') }}">
        {% with form=form, submit_label="Créer un compte" %}
            {% include "includes/form_fields.html" %}
        {% endwith %}
    </form>
{% endblock %}
```

**Remarques**: le formulaire utilise `form_fields.html`, et donc la template est quasiment identique à [app/templates/pages/icono_create.html](./apps/s6/create_user/app/templates/pages/icono_create.html).

## La route `user_create`

On a maintenant une classe formulaire et une template pour ce formulaire.

Maintenant, on va **créer la route `user_create` pour accéder au formulaire et pouvoir le soumettre**. Cette route se trouve dans [`app/routes/users.py`](./apps/s5/insert/app/routes/users.py)

```py
@app.route("/user/nouveau/", methods=["GET", "POST"])
def user_create():
    form = UserCreateForm()

    # form.validate_on_submit est True si:
    # - la requête est POST (on a soumis un formulaire)
    # - le formulaire est valide (wtforms a bien validé toutes les données fournies)
    if form.validate_on_submit():
        # on récupère les données et on les passe à create
        # `.data` permet de sélectionner la valeur fournie par l'utilisateur.ice
        user_name = form.user_name.data
        user_mail = form.user_mail.data
        user_password = form.user_password.data
        # `create` retourne:
        # - un booleen qui indique si la création réussi
        # - soit l'objet User crée, soit une liste d'erreurs
        success, data = User.create(
            user_name=user_name, 
            user_mail=user_mail, 
            user_password=user_password
        )
        # l'insert a réussi => rediriger sur la page d'accueil
        if success:
            flash("Compte utilisateur créé avec succès ! Vous pouvez maintenant vous connecter.", "success")
            return redirect("/")
        # l'insert a échoué => afficher les messages d'erreur.
        else: 
            data = "Les erreurs suivantes ont été repérées:" + ", ".join(data)
            flash(data, "error")
            return  render_template("pages/user_create.html", form=form, app_name=APP_NAME)
    return render_template("pages/user_create.html", form=form, app_name=APP_NAME)
```

**Explication de code**: comme pour toutes les routes de *create/update/delete*,
- **notre route accepte 2 méthodes HTTP**: cela est défini dans le `@app.route()` avec `methods=["GET", "POST"]`
    - `GET` est utilisé pour **accéder au formulaire** sans soumettre de données
    - `POST` est utilisé pour **soumettre le formulaire** (rappelez vous de la template, où on voit `<form method="POST">`)
- **la route a 2 branches correspondantes**
    - `if form.validate_on_submit()` confirme que on a envoyé une requête `POST` et que le formulaire est valide **=> on tente une insertion** avec `User.create()`
    - sinon, (c'est une requête `GET` pour accéder au formulaire ou les données ne sont pas valides), **on renvoie le formulaire** `user_create.html`.
- **`flash` est utilisé pour afficher les messages de succès ou d'erreur**. flash est une fonction Flask qui prend en 1er argument les messages à afficher, en 2e argument le statut du message (`success` ou `error`).

Et enfin, **on importe les routes de [`app/routes/users.py`](./apps/s6/create_user/app/routes/users.py)** dans [`app/app.py`](./apps/s6/create_user/app/app.py) en modifiant la dernière ligne du fichier:

```py
# l'ancien import était: `from app.routes import generic`
from app.routes import generic, users
```

## Tester le résultat

> **Lancer l'application `apps/s5/insert`**:
> ```py
> python ./apps/s5/insert/main.py
> ```

**Essayons de**:
- créer un nouveau `user`
- fournir des mauvaises données au formulaire pour voir comment elles sont gérées (mauvais format d'email, qui correspond déjà à un `user`...)



---

# Gestion d'utilisateurs

On peut maintenant créer des nouveaux `users`. Super ! Mais la gestion d'utilisateurs, ce n'est pas que pouvoir créer des comptes. C'est aussi:
- pouvoir **se connecter**
- **rester connecté.e** d'une page à l'autre
- **restreindre l'accès à certaines pages** seulement si on est connecté.e (dans notre cas: modifier la BDD seulement quand on est connecté.e).

## La gestion d'utilisateurs: Flask-Login

Flask-Login est un plugin qui gère la connexion d'utilisateurs. Pour l'utiliser, on doit:
- configurer un `LoginManager`
- augmenter `User` pour le rendre compatible avec Flask-Login
- définir un `user_loader` pour que l'appli Flask puisse déterminer les utilisateur.ice.s actuellement connecté.e.s

**Fichiers modifiés**:

```txt
apps/s6/login/
└── app
    ├── app.py        # on configure Flask-Login dans app.py
    └── models
        └── users.py  # on modifie la classe `User` pour que Flask-Login puisse interagir avec et on ajoute un `user_loader`
```

### Configurer le `LoginManager`

La première étape, c'est de compléter [`app/app.py`](./apps/s5/login/app/app.py):

```py
from flask_login import LoginManager

app = Flask(
    APP_NAME,
    template_folder=DIR_TEMPLATES, 
    static_folder=DIR_STATICS
)
# j'omets le reste de la config...
login_manager = LoginManager()
login_manager.init_app(app)

from app.routes import generic, users
```

`login_manager` est notre gestionnaire de connexions et il est totalement intégré à notre appli Flask grâce à `login_manager.init_app(app)`.

### Configurer l'`User`: `UserMixin`

Flask-Login a besoin que le modèle pour nos `Users` ait quelques propriétés bien définies pour fonctionner: `is_authenticated`, `is_active`, `is_anonymous`, `get_id` (qui permet de récupérer l'ID de l'utilisateur).

**Pas besoin de les définir à la main**: Flask-Login offre un `UserMixin` qui rajoute ces méthodes à notre modèle `User`. Pour ajouter le mixin, on modifie [`app/models/users.py`](./apps/s5/login/app/models/users.py):

```py
from flask_login import UserMixin

class User(db.Model, UserMixin):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
```

Et c'est tout ! Pour rappel, `User(db.Model, UserMixin)` signifie que **`User` hérite à la fois de `db.Model` et de `UserMixin`**. `UserMixin` définie les propriétés `is_authenticated`, `is_active`, `is_anonymous`, `get_id`m donc celles-ci deviennent accessibles depuis notre `User` (par exemple: `user.get_id()`).

### Définir un `user_loader`

Notre `LoginManager` défini dans `app/app.py` a besoin de pouvoir accéder à l'utilisateur actuellement connecté. **On définit donc une fonction qui permet d'accéder à un `User` par son ID**, toujours dans [`app/models/users.py`](./apps/s5/login/app/models/users.py):

```py
from app.app import login_manager

# ce décorateur signifie que la fonction `load_user` sera utilisée par le LoginManager pour accéder à l'user connecté.e
@login_manager.user_loader
def load_user(id_user: str):
    id_user = int(id_user)
    return db.session.get(User, id_user)
```

### Ce qui devient possible

Et voilà, Flask-Login est configuré.

Flask-Login offre des choses bien utile pour nous:
- une variable `current_user`, qui stocke l'`user` actuellement connecté.e dans une session (si il y a un `user` connecté)
- deux fonctions `login_user()` et `logout_user()` qui permettent de connecter/déconnecter un.e `user`.
- un décorateur `@login_required` qui permet de limiter l'accès à une route Flask aux utilisateur.ice.s connecté.es. 

## Login: se connecter

La logique à implémenter: 
- **une route** permet d'accéder à un formulaire pour se connecter
- pour se connecter à un compte existant, **l'utilisateur.ice fournit un mail et un mot de passe**
- **notre application vérifie** si ils correspondent à un `User` dans la base de données
- si oui, **on connecte l'utilisateur**

**Fichiers modifiés**:

```txt
apps/s6/login/
└── app
    ├── models
    │   ├── forms.py  # ajout du formulaire `UserLoginForm`
    │   └── users.py  # ajout d'une méthode pour identifier un `user` en fonction de son mail et mot de passe
    ├── routes
    │   └── users.py  # ajout d'une route pour le login
    └── templates
        └── pages
            └── user_login.html  # template avec un formulaire pour le login
```

### Formulaire Flask

Ensemble, ajoutons **un formulaire Flask `UserLoginForm`** pour se connecter. Pour rappel, on se connecte en utilisant son mail et son mdp (on peut utiliser `UserCreateForm` défini plus haut pour référence).


In [36]:
# le code ici

(La solution est dans `apps/s5/login/app/models/forms.py`)

### Template `user_login.html`

[`app/templates/pages/user_login.html`](./apps/s5/login/app/templates/pages/user_login.html) reprend la même structure que toutes nos pages-formulaires:

```html
{% block main_content %}
    <h1 class="title">Se connecter</h1>

    <!-- comment interprétez vous le contenu de `form` ? -->
    <form method="POST" action="{{ url_for('user_login') }}">
        {% with form=form, submit_label="Connexion" %} 
            {% include "includes/form_fields.html" %}
        {% endwith %}
    </form>
{% endblock %}
```

Il faut maintenant **ajouter la logique côté base de données et une route pour se connecter**.

### Identifier un `User`

Dans [`app/models/users.py](./apps/s6/login/app/models/users.py), **on augmente `User` en ajoutant la méthode `get_user_by_credentials`**. Elle vérifier si un `user` existe pour le mail et le MDP fournis. Voilà le processus:
- l'utilisateur.ice fournit son mail et son mot de passe
- on hashe le mot de passe
- on vérifie qu'il y a une ligne dans la base de données avec le bon mail et le bon hash de mot de passe.
- si oui, on retourne le `user`, sinon on retourne `None`.

```py
class User(db.Model, UserMixin):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
    
    @staticmethod
    def create(user_mail: str, user_name: str, user_password: str) -> Tuple[bool, Union["User", List[str]]]:
        ...
    
    @staticmethod
    def get_user_by_credentials(user_mail: str, user_password: str) -> Optional["User"]:
        """
        identifier un User par son mail et son mdp. 

        :returns: l'User si le mail et mdp sont valides, None sinon 
        """
        # retourne soit un `User`, soit None 
        user = db.session.execute(
            db.select(User).filter(User.user_mail == user_mail)
        ).scalars().first()
        # on vérifie que le `User` avec ce mail a bien le bon mot de passe
        if user and check_password_hash(user.user_password, user_password):
            return user
        # sinon, on retourne None
        return None
```

### Une route pour se connecter

Pour finir, on ajoute une route `/user/connexion` pour se connecter.

Voici la route dans [`app/routes/users.py`](./apps/s5/login/app/routes/users.py):

```py
from flask_login import login_user

# comment interpréter ce qui se passe dans cette route ?
@app.route("/user/connexion/", methods=["GET", "POST"])
def user_login():
    form = UserLoginForm()

    if current_user.is_authenticated:
        flash("Vous êtes déjà connecté.e", "success")
        return redirect("/")

    if form.validate_on_submit():
        user_mail = form.user_mail.data
        user_password = form.user_password.data
        user = User.get_user_by_credentials(
            user_mail=user_mail,
            user_password=user_password
        )
        if user:
            flash("Vous êtes maintenant connecté.e", "success")
            login_user(user)
            return redirect("/")
        else:
            flash("Identifiants incorrects", "error")
            return render_template("pages/user_login.html", form=form, app_name=APP_NAME)
        
    return render_template("pages/user_login.html", form=form, app_name=APP_NAME)
```

## Logout: se déconnecter

De la même manière que l'on a créé une route pour se connecter, on en crée une pour se déconnecter. C'est bien plus simple: on veut une simple requête `GET` à `user/deconnexion/`, et cette requête exécute la fonction Flask-Login `logout_user`:

On ajoute donc à  [`app/routes/users.py`](./apps/s5/login/app/routes/users.py) la fonction suivante. On remarque l'usage du décorateur `@login_required`: il n'est possible d'accéder à cette route que si on est connecté.e.

```py
@app.route("/user/deconnexion/")
@login_required
def user_logout():
    logout_user()
    flash("Vous êtes déconnecté.e")
    return redirect("/")
```

## Ajouter `@login_required` aux routes *create/update/delete*

Pour empêcher que des personnes non-connectées ne fassent n'importe quoi, on ajoute le décorateur `@login_required` à toutes les routes *create/update/delete* (sauf `user_create`, qui permet de créer son compte utilisateur).

On modifie donc les routes suivantes dans [`app/routes/generic.py`](./apps/s6/login/app/routes/generic.py) 

```py
@app.route("/iconographie/nouveau", methods=["GET", "POST"])
# on ajoute `login_required`: on ne peut créer une ressource icono que si on est connecté.e
@login_required
def icono_create():
    # ... le reste de la fonction ne change pas, je l'omets donc ici

@app.route("/iconographie/<int:id_icono>/modifier", methods=["GET", "POST"])
@login_required
def icono_update(id_icono: int):
    # ...

@app.route("/iconography/<int:id_icono>/supprimer", methods=["GET", "POST"])
@login_required
def icono_delete(id_icono: int):
    # ...
```

## Voir le résultat

Allons maintenant voir le résultat.

> **On lance l'application [`login`](./apps/s6/login/)**:
> ```py
> python apps/s6/login/main.py
> ```

Essayons de:
- créer un compte
- se connecter
- mettre les mauvais *credentials* (mail + mot de passe)
- accéder aux routes *create/update/delete* sans être connecté.e
- ...


---

# Tests

## Tester une appli ? Pourquoi ?

Pour le moment, on ajoute une fonctionnalité, on lance l'appli, on regarde si ça marche.

**Ajouter des tests permet d'automatiser cette vérification**: on écrit du code qui s'assure qu'une opération fonctionne comme prévu.

Mais si on peut le faire à la main, **pourquoi s'embêter** à écrire du code en plus ?
- parce que vérifier à la main, ça prend du temps et c'est faillible: on peut oublier de tester certaines conditions
- éviter les régressions (qui arrivent quand on ajoute une nouvelle fonctionnalité, mais que cet ajout casse une ancienne fonctionnalité)
- **ma raison principale**: parce que la complexité de votre code va rapidement dépasser votre "charge cognitive" (votre RAM mentale): avoir des tests permet de ne pas avoir à retenir tous les cas d'usage et les choses qui peuvent rater ou fonctionner pour une fonction, puisque votre test prouvera que le code marche comme prévu. **Écrire des tests permet donc de réduire votre charge cognitive**

**Quoi tester ?** 
- **idéalement**, pour chaque fonction, on devrait un test qui prouve qu'elle fonctionne (voir deux tests, un qui prouve qu'elle fonctionne comme prévu, l'autre que elle gère bien les erreurs). Mais ce n'est pas forcément réaliste
- **une bonne pratique c'est au moins de faire des tests pour toutes les routes qui modifient la base de données**: c'est souvent les routes les plus compliquées, et c'est celles qui auront un impact sur vos données à long terme.

**Quand écrire des tests ?** Le plus tôt possible, dès qu'on crée son appli. Rajouter des tests à une grosse appli alors qu'elle existe déjà, ça peut être très difficile.

**Comment écrire des tests ?** On va voir plus de détails en dessous, mais retenons qu'il faut:
- écrire des tests simples
- écrire 1 test par cas de figure.
    - on peut faire plusieurs tests par fonction ou par route.
    - au moins, c'est une bonne idée de tester un insert en base avec 1 test qui prouve qu'un insert qui devrait marcher marche, et qu'un insert qui devrait échouer échoue. 

## Tester une appli ? Comment ?

On va utiliser la librairie [Pytest](https://docs.pytest.org/en/stable/) pour faire nos tests. Python offre par défaut la librairie de tests [Unittest](https://docs.python.org/3/library/unittest.html), mais dans sa [documentation](https://flask.palletsprojects.com/en/stable/testing/), Flask conseille l'utilisation de Pytest (et sa syntaxe est plus clean que celle d'Unittest).

**Voici les fichiers qu'on crée ou modifie** dans l'application [`app_tests`](./apps/s6/app_tests/):

```txt
app_tests/
└── app
    ├── app.py               # on modifie la manière dont l'appli est configurée
    └── tests                # module contenant tous nos tests
        ├── __init__.py      
        ├── conftest.py      # la configuration de nos tests
        ├── test_basics.py   # quelques tests basiques
        └── test_users.py    # tests relatifs aux Users
```

**On lance nos tests avec la commande**:
```bash
cd apps/s6/app_tests
pytest  # ou `pytest -s` si on veut voir le résultat de nos prints.
```

## Configurer une appli pour des tests: `app.py`

Jusqu'à maintenant, notre application ne permet qu'une seule configuration. Par exemple, on il n'y a pas d'options pour se connecter à différentes bases de données.

Or, on a toujours besoin d'une configuration propre à nos tests: par exemple, on ne veut pas modifier notre base de données avec nos tests. On va donc **configurer conditionnellement notre `app`**.

Voilà le nouvel [`app/app.py`](./apps/s6/app_tests/app/app.py):

```py
def config_app(app):
    """
    configurations de l'app en fonction du contexte d'exécution. 

    on définit deux contextes grâce à la variable d'environnement 
    `FLASK_TESTING`.

    - FLASK_TESTING=True -> tests
    - FLASK_TESTING=False -> fonctionnement normal
    """
    
    # on cherche la variable d'env `FLASK_TESTING` avec `os.getenv`
    # si elle n'est pas définie, notre valeur par défault est "False"
    # `FLASK_TESTING` est définie par `app/tests/conftest.py`
    testing = os.getenv("FLASK_TESTING", "False").lower() == "true"

    if testing:
        app.config.update({
            "TESTING": True,
            "SQLALCHEMY_DATABASE_URI": "sqlite:///:memory:"
        })
    else:
        app.config.update({
            "TESTING": False,
            "SQLALCHEMY_DATABASE_URI": f"sqlite:///{PATH_DB}"
        })

    return app


app = Flask(
    APP_NAME,
    template_folder=DIR_TEMPLATES, 
    static_folder=DIR_STATICS
)
app.config["SECRET_KEY"] = SECRET_KEY
app = config_app(app)
db = SQLAlchemy(app)
login_manager = LoginManager()
login_manager.init_app(app)


from app.routes import generic, users
```

**Commentaire de code**:
- la fonction `config_app()` définit 2 configirations: une si `testing == True`, l'autre si `testing == False`
- on définit une variable **`testing` qui permet de choisir la bonne configuration**. 
    - `testing` est définie grâce à une variable d'environnement, `FLASK_TESTING`.
    - `FLASK_TESTING` est définie dans [`app/tests/conftest.py`](./apps/s6/app_tests/app/tests/conftest.py)
- `sqlite:///:memory:` permet de se connecter à une base de données qui n'existe que dans votre RAM (=> n'est jamais sauvegardée, est détruite à la fin des tests)

**À propos des variables d'environnement**: une variable d'environnement, c'est une valeur qui 
- est définie **au niveau de votre OS**. Elle peut donc être utilisée en dehors de votre application Python 
- peut être lue par des programmes pour **modifier leur comportement**. Par exemple, ici, `config_app` lit votre variable d'environnement et l'utilise pour définir app.config ;
- ressemble un peu à `app.config`, sauf que `app.config` existe uniquement dans votre application Flask ;
- sert souvent à stocker des valeurs que l'on veut modifier facilement et que **l'on ne veut pas mettre dans un dépôt en ligne**, par exemple : des identifiants de connexion ou le chemin vers une base de données.

À noter: notre `config_app` est un peu une solution de facilité. **En temps normal on a besoin de 3 configs au moins**: développement, tests et production. Il vaut mieux définir des classes, où chaque classe correspond à une config différente. Quand vous développerez une vraie appli, suivez [ce guide](https://flask.palletsprojects.com/en/stable/config/#configuration-best-practices).


## Le code de tests: `app/tests/`

Avec Pytest, les tests à exécuter sont trouvés automatiquement, mais notre code doit suivre une certaine structure:
- les tests sont stockés dans un dossier `tests/`
- chaque fichier contenant des tests à exécuter doit commencer par `test_`: `test_users.py`, par exemple
- `tests/conftest.py` permet de configurer les tests.

## Configuration des tests: `app/tests/conftest.py`

Avec Pytest, le fichier [`app/tests/conftest.py`](./apps/s6/app_tests/app/tests/conftest.py) permet de configutrer l'application et les tests à exécuter.

```py
import os

# pour éviter des problèmes de setup de l'appli, on définit FLASK_TESTING avant d'importer pytest
os.environ["FLASK_TESTING"] = "True"

import pytest

from app.app import app as flask_app, db as flask_db


@pytest.fixture(autouse=True)
def set_testing_env():
    yield
    # quand les tests sont finis, on supprime la vatriable d'env
    os.environ.pop("FLASK_TESTING", None)


@pytest.fixture()
def app():    
    # on désactive la validation CSRF pour pouvoir tester nos formulaires
    # (ATTENTION: ne jamais la désactiver sinon, ni en dev, ni en prod)
    flask_app.config['WTF_CSRF_ENABLED'] = False
    

    with flask_app.app_context():
        # db.create_all() est une commande SQLAlchemy qui permet de 
        # créer une base de données à partir des modèles qu'on a défini 
        # comme notre base de données de test est vide, on l'initialise
        flask_db.create_all()
        # yield est une forme particulière de `return`. 
        # ici, `yield` correspond à toute la durée d'exécution des tests.  
        yield flask_app
        # tout ce qui vient après le `yield` permet de nettoyer tout une fois que tous 
        # les tests se sont exécutés. ici, on supprime la base de données et son contenu.
        flask_db.session.remove()
        flask_db.drop_all()


@pytest.fixture()
def client(app):
    return app.test_client()


@pytest.fixture()
def db(app):
    yield flask_db     
```

**Commentaires de code**:
- `os.environ["FLASK_TESTING"] = "True"` permet de définir la variable d'env `FLASK_TESTING`, qui sera lue par `config_app()`
- `@pytest.fixture` définit une *fixture*, c'est à dire **des données qui permettent d'exécuter des tests**.
- dans notre cas, **les *fixtures* servent à définir**:
    - `app`: l'application Flask de test
    - `client`: un client qui pourra faire des requêtes à `app`
    - `db`: une base de données SQLAlchemy de test
- le `yield` est un peu complexe, retenons seulement qu'il permet de définir `app`, `client` et `db` pour que des applis puissent les utiliser.

## Des tests simples: `app/tests/test_basics.py`

### Les asserts

En Python, `assert` est un mot clé qui permet de vérifier qu'une condition est remplie.

Ci-dessous, les 2 conditions sont :
- `2*2 == 4`
- `2-2 == 4`

La première condition est `True`, il n'y a donc pas de problème. La 2e conditon est fausse, et cause donc une erreur (`AssertionError`).

Pytest utilise les `asserts`: au lieu de lancer une erreur, si une condition est `False`, le test échouera.

In [37]:
assert 2*2 == 4
assert 2-2 == 4

AssertionError: 

### Un premier test

Voici un exemple de test, dans [`app/tests/test_users.py`](./apps/s6/app_tests/app/tests/test_users.py):

```py
def test_config(app):
    assert app.config["TESTING"] is True
    assert "memory" in str(app.config["SQLALCHEMY_DATABASE_URI"])
```

On voit déjà que **un test est une fonction**. Cette fonction exécute du code et **vérifie qu'on obtient les résultats attendus**.
- **`assert` est utilisé par Pytest** pour vérifier que des conditions attendues sont remplies:
    - notre appli tourne bien en mode test (`TESTING`)
    - nous sommes bien connecté.e.s à la base de données de test.

**Commentaires**:
- la fonction est dans un fichier dont le nom commence par `test_` et le nom de la fonction commence par `test_`. **Pytest identifiera donc cette fonction comme un test**.
- **`app` correspondent à la *fixture* `app`** définie dans `conftest`. 

En fait, la valeur donnée à `app` dans `test_config` correspond au `yield` de chaque de la fonction `app()` qu'on a vue dans `conftest.py`.

### Tester une route *create*

Le test ci-dessous se trouve dans [`app/tests/test_users.py`](./apps/s6/app_tests/app/tests/test_users.py) et permet de tester la route `user_create`:

```py
def test_create_user_ok(app, db, client):
    # que se passe-t-il ici ?
    response = client.post(
        '/user/nouveau/',
        data={
            'user_name': 'testuser',
            'user_mail': 'test@example.com',
            'user_password': 'securepassword123'
        }
    )

    # si la création d'utilisateurs a fonctionné, on est redirigé.e vers la page d'accueil.
    # le statut HTTP 302 fonctionne pour la redirection
    assert response.status_code == 302
    # que vérifie-t'on ici ?
    assert response.location == '/'

    # et ici ?
    with app.app_context():
        user = db.session.execute(db.select(User).filter_by(user_name='testuser')).scalars().first()
        assert user.user_mail == 'test@example.com'
```

On remarque que notre test prend en arguments `app`, `db` et `client`, qui ont tous les trois été définis comme *fixtures* dans `conftest.py`.

Regardons maintenant ce qui se passe dans `test_create_user_error`, dans [`app/tests/test_users.py`](./apps/s6/app_tests/app/tests/test_users.py).

> **On lance toute la suite de tests**:
> ```bash
> cd apps/s6/app_tests/
> pytest
> ```

## Et un test qui rate ?

On crée une nouvelle appli [`app_tests_failure`](./apps/s6/app_tests_failure/). Le seul changement, c'est qu'on ajoute un test qui va rater pour que vous voyez ce qui s'affiche dans le terminal:

```py
# ce test va planter
def test_home_page(client):
    response = client.get('/route/qui/nexiste/pas')
    # on va se manger un status code 404 => l'assert ci-dessous rate => nos tests ne réussissent pas
    assert response.status_code == 200
```

> **On lance toute la suite de pour voir le résultat**:
> ```bash
> cd apps/s6/app_tests_failure/
> pytest
> ```



---

# Écrire une API

Notre dernier challenge avant d'être officiellement des champion.ne.s de Flask: écrire une API.

## API ?

Une API (*application programming interface*, ou interface de programmation) définit un protocole pour que **deux programmes informatiques communiquent entre eux**. C'est un concept qu'on retrouve un peu partout en informatique.

En développement Web, **une API désigne une version du site conçue pour être utilisée par des machines, et non par des utilisateur.ice.s humain.e.s**. Essentiellement, une API définit **des routes qui renvoient du JSON à la place de documents HTML**. Et c'est tout.

Le JSON est un format de données structuré, plus léger que le HTML et donc plus facile à utiliser par des programmes informatiques. Une API, c'est donc très utile quand on veut récupérer des données d'un site: on écrit un script qui fait des requêtes sur ce site, et on récupère des données déjà structurées.

## *JSONifier* un modèle

Le JSON, c'est exactement comme les listes et les dictionnaires en Python.

Si on veut que notre appli renvoie du JSON, il faut **représenter nos tables de base de données en JSON**. Dans ce cas:
- **une instance est un `dict`** (p.ex. une ressource iconographique)
- **une collection d'objets est une `list` de `dicts`** (par exemple, toutes les lignes de la table `Iconography`)

On instancie les modèles nécessaires pour JSONifier `Iconography`, puis **on crée la fonction `icono_item_to_dict`** qui prend en entrée un objet `Iconography` et produit une représentation de celui-ci en `Dict`.

In [39]:
from typing import Optional, Union, List, Dict, Tuple

from sqlalchemy import ForeignKey, JSON
from sqlalchemy.orm import Mapped, mapped_column, relationship

# on redéfinit tous les modèles Iconography, Author et Place

class IconographyPlace(db.Model):
    __tablename__ = "iconography_place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    id_iconography: Mapped[int] = mapped_column(ForeignKey("iconography.id"))
    id_place: Mapped[int] = mapped_column(ForeignKey("place.id"))


class IconographyTheme(db.Model):
    __tablename__ = "iconography_theme"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    id_iconography: Mapped[int] = mapped_column(ForeignKey("iconography.id"))
    id_theme: Mapped[int] = mapped_column(ForeignKey("theme.id"))


class Author(db.Model):
    __tablename__ = "author"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    author_name: Mapped[str] = mapped_column(unique=True)

    iconography: Mapped[List["Iconography"]] = relationship(
        back_populates="author", 
    )


class Theme(db.Model):
    __tablename__ = "theme"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    theme_name: Mapped[str] = mapped_column(unique=True)
    richelieu_url: Mapped[str] = mapped_column(unique=True)

    iconography: Mapped[List["Iconography"]] = relationship(
        secondary=IconographyTheme.__table__,
        back_populates="theme"
    )


class Place(db.Model):
    __tablename__ = "place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    address: Mapped[Optional[str]]
    richelieu_url: Mapped[str] = mapped_column(unique=True)
    loc: Mapped[Dict] = mapped_column(JSON)
    plot: Mapped[Dict] = mapped_column(JSON)
    date_lower: Mapped[int]
    date_upper: Mapped[int]

    iconography: Mapped[List["Iconography"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="place"
    )


class Iconography(db.Model):
    __tablename__ = "iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    id_author: Mapped[Optional[int]] = mapped_column(ForeignKey("author.id"))
    
    author: Mapped[Optional["Author"]] = relationship(back_populates="iconography")
    theme: Mapped[List["Theme"]] = relationship(
        secondary=IconographyTheme.__table__,
        back_populates="iconography"
    )
    place: Mapped[List["Place"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="iconography"
    )


In [42]:
# maintenant, on écrit une fonciton qui permet de représenter un objet `Iconography` en JSON

def icono_item_to_dict(icono_item: Iconography) -> Dict:
    """
    représentation d'une ressource iconographique en JSON

    NOTE: cette fonction est simple mais n'est pas petrformante 
    et va être très lente si on veut l'utiliser sur beaucoup d'objets 
    icono à la suite 
    """
    #  icono_dict est une représentation de `icono_item` sous forme de dictionnaire
    icono_dict = {
        "id": icono_item.id,
        "title": icono_item.title,
        "iiif_image_url": icono_item.iiif_image_url,
        "iiif_manifest_url": icono_item.iiif_manifest_url,
        "source_url": icono_item.source_url,
        "richelieu_url": icono_item.richelieu_url,
        "date_lower": icono_item.date_lower,
        "date_lower": icono_item.date_lower,
        "date_lower": icono_item.date_lower,
        "date_upper": icono_item.date_upper or None,
        "institution": icono_item.institution,
    }

    # on le complète avec les relations
    if icono_item.author:
        icono_dict["author"] = icono_item.author.author_name
    if icono_item.place:
        places = []
    for place in icono_item.place:
        places.append(place.address)
        icono_dict["places"] = places
    if icono_item.theme:
        themes = []
        for theme in icono_item.theme:
            themes.append(theme.theme_name)
    icono_dict["themes"] = themes
    return icono_dict

# et on le teste ensemble. on veut tester:
# - la représentation en dict de l'Iconography avec l'ID 999
with app.app_context():
    icono_id = 999
    # à vous de jouer


## Les routes API

Le fonctionnement de notre API reprend exactement celui de notre application: on voudrait 2 routes pour chaque modèle: 
- une route pour **voir un catalogue de toutes les ressources**, 
- une route pour **voir une seule ressource**  

On ne crée ces routes que pour la table `Iconography`.

**Voici le contenu de [`app/routes/api.py`](./apps/s6/api/app/routes/api.py)**, le fichier qui contient notre API.

```py
from flask import jsonify

@app.route("/api/iconographie")
def api_icono_index():
    # on récupère tout notre corpus iconographique
    icono_corpus = db.session.execute(
        db.select(Iconography)
    ).scalars()

    # notre variable de sortie
    icono_list = []
    for icono_item in icono_corpus:
        icono_dict = icono_item_to_dict(icono_item)
        # on l'ajoute à icono_list
        icono_list.append(icono_dict)
    
    return jsonify(icono_list)


@app.route("/api/iconographie/<int:id_icono>")
def api_icono_main(id_icono: int):
    # on récupère tout notre corpus iconographique
    icono_item = db.get_or_404(Iconography, id_icono)
    icono_dict = icono_item_to_dict(icono_item)
        
    # on retourne le dict    
    return jsonify(icono_dict)
```

**Remarques**
- comme dit au dessus, **on définit 2 routes**, l'une pour voir toutes les ressources icono, l'autre pour en voir une seule (en fonction de son ID)
- on utilise `icono_item_to_dict` pour **représenter chaque objet par un `dict`**
- **dans le `return`, on utilise `jsonify`**:  une fonction Flask qui prend une `list` ou un `dict` et produit une réponse HTTP en JSON (sorte d'équivalent de `render_template` utilisé jusqu'alors)

> **On lance l'application [`api`](./apps/s6/api/)**
> ```py
> python ./apps/s6/api/main.py
> ```

On va voir les URLs suivantes. Que voit-on ?
- [`localhost:5000/api/iconographie`](http://localhost:5000/api/iconographie)
- [`localhost:5000/api/iconographie/1`](http://localhost:5000/api/iconographie/1)
- [`localhost:5000/api/iconographie/9999999`](http://localhost:5000/api/iconographie/9999999)

Pour finir, j'ai aussi créé le fichier [`app/tests/test_api.py`](./apps/s6/api/app/tests/test_api.py) qui contient des tests pour l'API. Vous pouvez aller voir les tests pour plus d'exemples de tests.

Et voilà !


---

# En conclusion

## Choses vues

Pendant ces cours, on aura appris à faire du développement Web en Python, mais aussi à connecter ensemble plein de petites choses que vous aurez apprises au cours de votre année de TNAH.

Les librairies suivantes ont été abordées:
- **Flask**, pour le développement Web
- **SQLAlchemy**, pour interagir avec des bases de données en Python
- **Jinja**, générer du HTML à partir de données Python
- **WTForms**, pour créer et valider des formulaires 
- **Pytest**, pour écrire des tests

On aura appris:
- les fondamentaux du développement Web en Python (routes, templates, formats de réponse, gestion d'erreurs, HTTP, APIs...)
- faire du CRUD sur une base SQL en Python
- tester: comment, quand, quoi ?
- et peut-être le plus important: **structurer et naviguer une "vraie" codebase**, organisée en plusieurs modules

## Pour aller plus loin: frontend/backend

On sait maintenant développer un site Web... Comme en 2006:
- **toutes les pages sont "synchrones"**: il faut recharger l'intégralité de la page pour charger des données
- **tout notre HTML est généré en Python**, via des Jinja. 1 page = 1 template.

Aujourd'hui, **la plupart des sites sont "asynchrones"**: ils permettent de reçevoir des données sans avoir à recharger l'intégralité de la page (sur instagram ou dans vos mails, vous n'avez pas besoin de recharger la page pour voir qu'on a reçu un message).

Ce type de fonctionnement est permis par une architecture en 2 parties:
- **un backend** qui gère l'interaction avec la base de données et expose une API. **Les frameworks Web de Python** sont parfaits pour faire du Backend: Flask, Django, FastAPI.
- **un frontend** dynamique qui gère la mise en page (structuration HTML): il interagit avec l'API via des requêtes HTTP, reçoit des données et les structures en HTML. **Les frontends sont écrits avec des frameworks spécialisés JavaScript**: [Vue](https://vuejs.org/), [Svelte](https://svelte.dev/), [React](https://react.dev/).

Cette architecture permet que le frontend fasse plein de requêtes au backend pour **modifier seulement certaines parties de la page** (par exemple, reçevoir des messages ou des notifications). Cela permet aussi de d'éćrire du code plus modulaire et de mieux [séparer les préoccupations](https://fr.wikipedia.org/wiki/S%C3%A9paration_des_pr%C3%A9occupations): 
- le backend fait le travail difficile de gestion de données 
- le frontend se consacre à leur mise en forme

Apprendre à utiliser des frontends JS demande d'avoir déjà des bonnes bases de Javascript. C'est hors du domaine de ce cours, mais si vous développez une appli, vous devrez probabelement ajouter de l'asynchronicité c'est bien de savoir au moins que les frameworks existent (et aussi, les frameworks de frontend, c'est très fun !).  